<div style = "background-color:#36A6A6; padding:5px; text-align:left; border-left:5px solid #00ced1;">
    <h1 style = "color:white; font-family:calibri; margin:0;">Retail Sales Analytics Dashboard</hi>
    <p style = "color:white; font-size:16px; margin:0;">Business Intelligence & Reporting</p>
</div>

### <span style="color:#36A6A6">STEP 1 — Create a safe SQLAlchemy engine for PostgreSQL</span>

In [50]:
# STEP 1 — Create a safe SQLAlchemy engine for PostgreSQL
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# IMPORTANT: for security prefer environment variables; replace for quick runs
db_user = os.getenv("PGUSER", "postgres")
db_password = os.getenv("PGPASSWORD", "XXXXxxxxxXX")  # consider setting as env var
db_host = os.getenv("PGHOST", "localhost")
db_port = os.getenv("PGPORT", "5432")
db_name = os.getenv("PGDB", "Retail_Sales_DB")

connection_url = URL.create(
    "postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database=db_name
)

engine = create_engine(connection_url)
print("Engine created. Test connection below...")

# optional quick test
with engine.connect() as conn:
    r = conn.execute(text("SELECT 1")).scalar()
    print("Test query returned:", r)


Engine created. Test connection below...
Test query returned: 1


### <span style="color:#36A6A6"> STEP 2 — Load CSV safely (parse Date column) </span>

In [51]:
# STEP 2 — Load CSV safely (parse Date column)
import pandas as pd

df = pd.read_csv(
    "Retail_Transactions_Dataset.csv",
    low_memory=False,
    parse_dates=['Date'],   # your CSV uses 'Date' column (from sample data provided)
    dayfirst=False
)

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()


Loaded rows: 1000000
Columns: ['Transaction_ID', 'Date', 'Customer_Name', 'Product', 'Total_Items', 'Total_Cost', 'Payment_Method', 'City', 'Store_Type', 'Discount_Applied', 'Customer_Category', 'Season', 'Promotion']


,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000000,2022-01-21 06:27:29,Stacey Price,"['Ketchup', 'Shaving Cream', 'Light Bulbs']",3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN
1,1000000001,2023-03-01 13:01:21,Michelle Carlson,"['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'P...",2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One)
2,1000000002,2024-03-21 15:37:04,Lisa Graves,['Spinach'],6,41.49,Credit Card,Houston,Department Store,True,Professional,Winter,NaN
3,1000000003,2020-10-31 09:59:47,Mrs. Patricia May,"['Tissues', 'Mustard']",1,39.34,Mobile Payment,Chicago,Pharmacy,True,Homemaker,Spring,NaN
4,1000000004,2020-12-10 00:59:59,Susan Mitchell,['Dish Soap'],10,16.42,Debit Card,Houston,Specialty Store,False,Young Adult,Winter,Discount on Selected Items


### <span style="color:#36A6A6"> Checking the general table structure and missing values</span>

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   Transaction_ID     1000000 non-null  int64         
 1   Date               1000000 non-null  datetime64[ns]
 2   Customer_Name      1000000 non-null  object        
 3   Product            1000000 non-null  object        
 4   Total_Items        1000000 non-null  int64         
 5   Total_Cost         1000000 non-null  float64       
 6   Payment_Method     1000000 non-null  object        
 7   City               1000000 non-null  object        
 8   Store_Type         1000000 non-null  object        
 9   Discount_Applied   1000000 non-null  bool          
 10  Customer_Category  1000000 non-null  object        
 11  Season             1000000 non-null  object        
 12  Promotion          666057 non-null   object        
dtypes: bool(1), datetime64[ns](1

### <span style="color:#36A6A6">Summing up the missing values</span> 

In [53]:
df.isnull().sum()

Transaction_ID            0
Date                      0
Customer_Name             0
Product                   0
Total_Items               0
Total_Cost                0
Payment_Method            0
City                      0
Store_Type                0
Discount_Applied          0
Customer_Category         0
Season                    0
Promotion            333943
dtype: int64

### <span style="color:#36A6A6">Normalize column names to consistent casing and names</span>

In [54]:
# STEP 3 — Normalize column names to consistent casing and names
df.columns = [c.strip() for c in df.columns]  # trim spaces
# If you prefer lowercase names for all internal processing:
df.columns = [c.strip() for c in df.columns]  # keep original case for readability
# But for safety we will reference columns by exact names shown: Date, Product, etc.

# Clean Promotion: set missing -> "No Promotion", strip stray characters
df['Promotion'] = df['Promotion'].fillna('No Promotion').astype(str).str.strip()
df.loc[df['Promotion']=='', 'Promotion'] = 'No Promotion'
df['Promotion'] = df['Promotion'].str.replace(r"[\[\]']", "", regex=True).str.strip()

# Clean Payment_Method, Customer_Category, Store_Type minimal normalization
df['Payment_Method'] = df['Payment_Method'].fillna('Unknown').astype(str).str.strip()
df['Customer_Category'] = df['Customer_Category'].fillna('Unknown').astype(str).str.strip()
df['Store_Type'] = df['Store_Type'].fillna('Unknown').astype(str).str.strip()

# Date already parsed; ensure dtype
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# quick null checks
print("Null counts (selected):")
print(df[['Date','Product','Total_Items','Total_Cost']].isnull().sum())


Null counts (selected):
Date           0
Product        0
Total_Items    0
Total_Cost     0
dtype: int64


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   Transaction_ID     1000000 non-null  int64         
 1   Date               1000000 non-null  datetime64[ns]
 2   Customer_Name      1000000 non-null  object        
 3   Product            1000000 non-null  object        
 4   Total_Items        1000000 non-null  int64         
 5   Total_Cost         1000000 non-null  float64       
 6   Payment_Method     1000000 non-null  object        
 7   City               1000000 non-null  object        
 8   Store_Type         1000000 non-null  object        
 9   Discount_Applied   1000000 non-null  bool          
 10  Customer_Category  1000000 non-null  object        
 11  Season             1000000 non-null  object        
 12  Promotion          1000000 non-null  object        
dtypes: bool(1), datetime64[ns](1

In [56]:
df.isnull().sum()

Transaction_ID       0
Date                 0
Customer_Name        0
Product              0
Total_Items          0
Total_Cost           0
Payment_Method       0
City                 0
Store_Type           0
Discount_Applied     0
Customer_Category    0
Season               0
Promotion            0
dtype: int64

### <span style="color:#36A6A6">Convert product column to real Python lists if stored as string representations</span>

In [57]:
# STEP 4 — Convert Product column to real Python lists if stored as string representations
import ast

def to_list(val):
    if isinstance(val, list):
        return [str(x).strip() for x in val]
    if pd.isna(val):
        return []
    s = str(val).strip()
    # if it looks like a list string: "['a','b']" or "['a', 'b']"
    if s.startswith('[') and s.endswith(']'):
        try:
            return [str(x).strip() for x in ast.literal_eval(s)]
        except Exception:
            # fallback: split by comma
            return [p.strip() for p in s.strip("[]").split(',') if p.strip()]
    # if comma-separated single string
    if ',' in s:
        return [p.strip() for p in s.split(',') if p.strip()]
    if s=='':
        return []
    return [s]

df['Product'] = df['Product'].apply(to_list)

# quick check
print("Sample Product lists (first 5):")
print(df['Product'].head(5).tolist())


Sample Product lists (first 5):
[['Ketchup', 'Shaving Cream', 'Light Bulbs'], ['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'Potatoes'], ['Spinach'], ['Tissues', 'Mustard'], ['Dish Soap']]


### <span style="color:#36A6A6">Explode the product lists into one row per product</span>

In [60]:
# STEP 5 — Explode the product lists into one row per product
df_exploded = df.explode('Product').reset_index(drop=True)

# Clean product strings
df_exploded['Product'] = df_exploded['Product'].astype(str).str.strip()
# If any empty product rows (from transactions with no product), drop them
df_exploded = df_exploded[df_exploded['Product']!=''].copy()

# Add a product_line_quantity and allocate revenue among products
# Each exploded row represents 1 unit of that product in the basket by default
df_exploded['line_quantity'] = 1

# Number-of-items-in-original-transaction (after original parsing)
# If df had Total_Items column representing basket size, prefer that, otherwise compute:
if 'Total_Items' in df_exploded.columns and pd.api.types.is_numeric_dtype(
    df_exploded['Total_Items']):
    df_exploded['basket_size'] = df_exploded['Total_Items']
else:
    # compute by grouping original Transaction_ID
    basket_size = df_exploded.groupby('Transaction_ID')['Product'].transform('count')
    df_exploded['basket_size'] = basket_size

# Allocate revenue evenly across items if Total_Cost exists
if 'Total_Cost' in df_exploded.columns:
    # protect division by zero
    df_exploded['basket_size'] = df_exploded['basket_size'].replace({0:1})
    df_exploded['line_revenue'] = df_exploded['Total_Cost'].astype(float) 
    df_exploded['basket_size'].astype(float)
else:
    df_exploded['line_revenue'] = 0.0

print("Exploded rows:", len(df_exploded))
df_exploded.head(10)


Exploded rows: 3000343


,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion,line_quantity,basket_size,line_revenue
0,1000000000,2022-01-21 06:27:29,Stacey Price,Ketchup,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,No Promotion,1,3,71.65
1,1000000000,2022-01-21 06:27:29,Stacey Price,Shaving Cream,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,No Promotion,1,3,71.65
2,1000000000,2022-01-21 06:27:29,Stacey Price,Light Bulbs,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,No Promotion,1,3,71.65
3,1000000001,2023-03-01 13:01:21,Michelle Carlson,Ice Cream,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),1,2,25.93
4,1000000001,2023-03-01 13:01:21,Michelle Carlson,Milk,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),1,2,25.93
5,1000000001,2023-03-01 13:01:21,Michelle Carlson,Olive Oil,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),1,2,25.93
6,1000000001,2023-03-01 13:01:21,Michelle Carlson,Bread,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),1,2,25.93
7,1000000001,2023-03-01 13:01:21,Michelle Carlson,Potatoes,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),1,2,25.93
8,1000000002,2024-03-21 15:37:04,Lisa Graves,Spinach,6,41.49,Credit Card,Houston,Department Store,True,Professional,Winter,No Promotion,1,6,41.49
9,1000000003,2020-10-31 09:59:47,Mrs. Patricia May,Tissues,1,39.34,Mobile Payment,Chicago,Pharmacy,True,Homemaker,Spring,No Promotion,1,1,39.34


### <span style="color:#36A6A6">Re-exploding the product list safely to allocate revenue equally across product line</span>

In [62]:
# STEP X — Re-explode product list safely


# STEP X — Explode the product lists into one row per product
df_exploded = df.explode('Product').reset_index(drop=True)

# Clean product strings
df_exploded['Product'] = df_exploded['Product'].astype(str).str.strip()
# If any empty product rows (from transactions with no product), drop them
df_exploded = df_exploded[df_exploded['Product']!=''].copy()
# Clean product strings
df_exploded['Product'] = df_exploded['Product'].astype(str).str.strip()
# If any empty product rows (from transactions with no product), drop them
df_exploded = df_exploded[df_exploded['Product']!=''].copy()

# 1. Ensure Product column is always a string before splitting
df_exploded['Product'] = df_exploded['Product'].astype(str)

# 2. Split multiple items
df_exploded['Product'] = df_exploded['Product'].str.split(',')

# 3. Explode into multiple rows
df_exploded = df_exploded.explode('Product')

# 4. Convert again to string and strip spaces
df_exploded['Product'] = df_exploded['Product'].astype(str).str.strip()

# 5. Remove rows where 'Product' is null, empty, or 'nan'
df_exploded = df_exploded[
    df_exploded['Product'].notna() &
    (df_exploded['Product'] != '') &
    (df_exploded['Product'].str.lower() != 'nan')
]

# 6. Count # products per transaction
df_exploded['product_count'] = df_exploded.groupby('Transaction_ID')['Product'].transform('count')

# 7. Allocate revenue equally across products
df_exploded['line_revenue'] = df_exploded['Total_Cost'] / df_exploded['product_count']

# 8. Each item is quantity 1
df_exploded['line_quantity'] = 1

print("Exploded rows:", len(df_exploded))
df_exploded.head()


Exploded rows: 3000343


,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion,product_count,line_revenue,line_quantity
0,1000000000,2022-01-21 06:27:29,Stacey Price,Ketchup,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,No Promotion,3,23.883333,1
1,1000000000,2022-01-21 06:27:29,Stacey Price,Shaving Cream,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,No Promotion,3,23.883333,1
2,1000000000,2022-01-21 06:27:29,Stacey Price,Light Bulbs,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,No Promotion,3,23.883333,1
3,1000000001,2023-03-01 13:01:21,Michelle Carlson,Ice Cream,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),5,5.186000,1
4,1000000001,2023-03-01 13:01:21,Michelle Carlson,Milk,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),5,5.186000,1


### <span style="color:#36A6A6">Build dimension tables, ensure deterministic durable keys(surrogates)</span>

In [63]:
# STEP 6 — Build dimension tables, ensure deterministic durable keys (surrogates)
# CUSTOMER DIM
df_customer = (df_exploded[['Customer_Name','Customer_Category']]
               .drop_duplicates()
               .reset_index(drop=True))
df_customer['customer_id'] = df_customer.index + 1

# PRODUCT DIM (unique product list from exploded)
df_product = (df_exploded[['Product']]
              .drop_duplicates()
              .reset_index(drop=True))
df_product['product_id'] = df_product.index + 1

# STORE DIM (City + Store_Type combination)
df_store = (df_exploded[['City','Store_Type']]
            .drop_duplicates()
            .reset_index(drop=True))
df_store['store_id'] = df_store.index + 1

# DATE DIM (unique calendar dates)

df_date = (df_exploded[['Date','Season']]
           .drop_duplicates()
           .reset_index(drop=True))

# BUILD DIM_DATE TABLE WITH DAY & MONTH NAMES

# Start from df_exploded (line-level data)
df_date = (
    df_exploded[['Date', 'Season']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Ensure Date is proper datetime64
df_date['Date'] = pd.to_datetime(df_date['Date'])

# Create the surrogate key in YYYYMMDD format
df_date['date_id'] = df_date['Date'].dt.strftime('%Y%m%d').astype(int)

# Numeric components
df_date['year'] = df_date['Date'].dt.year
df_date['month'] = df_date['Date'].dt.month
df_date['day'] = df_date['Date'].dt.day

# Final reorder for neatness
df_date = df_date[[
    'date_id',
    'Date',
    'Season',
    'year',
    'month',
    'day'
]]

df_date.head()


# PROMOTION DIM
df_promotion = (df_exploded[['Promotion']]
                .drop_duplicates()
                .reset_index(drop=True))
df_promotion['promotion_id'] = df_promotion.index + 1

# PAYMENT METHOD DIM
df_payment_method = (df_exploded[['Payment_Method']]
                     .drop_duplicates()
                     .reset_index(drop=True))
df_payment_method['payment_method_id'] = df_payment_method.index + 1

# Show counts
print("dim sizes: customers={}, products={}, stores={}, dates={}, promos={}, payments={}".
      format(
    len(df_customer), len(df_product), len(df_store), len(df_date), len(df_promotion), 
    len(df_payment_method)
))


dim sizes: customers=676645, products=81, stores=60, dates=999093, promos=3, payments=4


### <span style="color:#36A6A6">Merge dims into the fact (df_exploded)</span>

In [64]:
# STEP 7 — Merge dims into the fact (df_exploded)
df_fact = df_exploded.copy()

# Merge customer_id
df_fact = df_fact.merge(df_customer, on=['Customer_Name','Customer_Category'], how='left')

# Merge product_id
df_fact = df_fact.merge(df_product, on='Product', how='left')

# Merge store_id (join on City + Store_Type)
df_fact = df_fact.merge(df_store, on=['City','Store_Type'], how='left')

# Merge date_id (ensure both sides are datetime64)
df_fact['Date'] = pd.to_datetime(df_fact['Date'])
df_date['Date'] = pd.to_datetime(df_date['Date'])
df_fact = df_fact.merge(df_date[['Date','date_id']], on='Date', how='left')

# Merge promotion_id
df_fact = df_fact.merge(df_promotion, on='Promotion', how='left')

# Merge payment_method_id
df_fact = df_fact.merge(df_payment_method, on='Payment_Method', how='left')

# Keep discount as boolean
df_fact['Discount_Applied'] = df_fact['Discount_Applied'].astype(bool)

# Create final fact columns: transaction_id, date_id, customer_id, product_id, store_id, promotion_id, 
# payment_method_id, discount_applied, quantity, revenue
df_fact_final = df_fact.rename(columns={
    'Transaction_ID':'transaction_id',
    'date_id':'date_id',
    'customer_id':'customer_id',
    'product_id':'product_id',
    'store_id':'store_id',
    'promotion_id':'promotion_id',
    'payment_method_id':'payment_method_id',
    'Discount_Applied':'discount_applied',
    'line_quantity':'quantity',
    'line_revenue':'revenue'
})

cols_needed = ['transaction_id','date_id','customer_id','product_id','store_id',
               'promotion_id','payment_method_id','discount_applied','quantity','revenue']
# check presence
missing = [c for c in cols_needed if c not in df_fact_final.columns]
if missing:
    print("WARNING: missing columns in final fact:", missing)
else:
    df_fact_final = df_fact_final[cols_needed].copy()

print("Final fact rows:", len(df_fact_final))
df_fact_final.head()


Final fact rows: 3016917


,transaction_id,date_id,customer_id,product_id,store_id,promotion_id,payment_method_id,discount_applied,quantity,revenue
0,1000000000,20220121,1,1,1,1,1,True,1,23.883333
1,1000000000,20220121,1,2,1,1,1,True,1,23.883333
2,1000000000,20220121,1,3,1,1,1,True,1,23.883333
3,1000000001,20230301,2,4,2,2,2,True,1,5.186000
4,1000000001,20230301,2,5,2,2,2,True,1,5.186000


### <span style="color:#36A6A6">Basic validation checks</span>

In [65]:
# STEP 8 — Basic validation checks
print("Unique transactions in fact:", df_fact_final['transaction_id'].nunique())
print("Rows in fact:", len(df_fact_final))
print("Sample aggregated revenue check (per transaction):")
sample_tx = df_fact_final.groupby('transaction_id')['revenue'].sum().reset_index().rename(
    columns={'revenue':'sum_revenue'})
# join back to original to compare with Total_Cost (if present)
if 'Total_Cost' in df.columns:
    orig_costs = df[['Transaction_ID','Total_Cost']].drop_duplicates().rename(columns={
        'Transaction_ID':'transaction_id','Total_Cost':'total_cost'})
    check = sample_tx.merge(orig_costs, on='transaction_id', how='left')
    check['diff'] = check['sum_revenue'] - check['total_cost']
    print(check.head())
else:
    print("Original Total_Cost not present for comparison")

# Check for null foreign keys
for fk in ['date_id','customer_id','product_id','store_id','promotion_id',
           'payment_method_id']:
    nulls = df_fact_final[fk].isnull().sum()
    print(f"{fk} nulls: {nulls}")


Unique transactions in fact: 1000000
Rows in fact: 3016917
Sample aggregated revenue check (per transaction):
   transaction_id  sum_revenue  total_cost  diff
0      1000000000        71.65       71.65   0.0
1      1000000001        25.93       25.93   0.0
2      1000000002        41.49       41.49   0.0
3      1000000003        39.34       39.34   0.0
4      1000000004        16.42       16.42   0.0
date_id nulls: 0
customer_id nulls: 0
product_id nulls: 0
store_id nulls: 0
promotion_id nulls: 0
payment_method_id nulls: 0


### <span style="color:#36A6A6">Create table Data Definition Language (DDL)</span>

In [66]:
# STEP 9 — Create table DDL (run once). Adjust schema name if needed.
ddl_statements = [
"""
CREATE SCHEMA IF NOT EXISTS retail;
""",
"""
CREATE TABLE IF NOT EXISTS retail.dim_customer (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    customer_category TEXT
);
""",
"""
CREATE TABLE IF NOT EXISTS retail.dim_product (
    product_id INTEGER PRIMARY KEY,
    product TEXT
);
""",
"""
CREATE TABLE IF NOT EXISTS retail.dim_store (
    store_id INTEGER PRIMARY KEY,
    city TEXT,
    store_type TEXT
);
""",
"""
CREATE TABLE IF NOT EXISTS retail.dim_date (
    date_id INTEGER PRIMARY KEY,
    date DATE,
    season TEXT,
    year INTEGER,
    month INTEGER,
    day INTEGER
);
""",
"""
CREATE TABLE IF NOT EXISTS retail.dim_promotion (
    promotion_id INTEGER PRIMARY KEY,
    promotion TEXT
);
""",
"""
CREATE TABLE IF NOT EXISTS retail.dim_payment_method (
    payment_method_id INTEGER PRIMARY KEY,
    payment_method TEXT
);
""",
"""
CREATE TABLE IF NOT EXISTS retail.fact_sales (
    fact_id BIGSERIAL PRIMARY KEY,
    transaction_id BIGINT,
    date_id INTEGER REFERENCES retail.dim_date(date_id),
    customer_id INTEGER REFERENCES retail.dim_customer(customer_id),
    product_id INTEGER REFERENCES retail.dim_product(product_id),
    store_id INTEGER REFERENCES retail.dim_store(store_id),
    promotion_id INTEGER REFERENCES retail.dim_promotion(promotion_id),
    payment_method_id INTEGER REFERENCES retail.dim_payment_method(payment_method_id),
    discount_applied BOOLEAN,
    quantity INTEGER,
    revenue NUMERIC(12,2)
);
"""
]

with engine.begin() as conn:
    for s in ddl_statements:
        conn.exec_driver_sql(s)
print("DDL executed (schema and tables created).")


DDL executed (schema and tables created).


### <span style="color:#36A6A6">Converting Customer columns into lower case and uploading to PostgreSQL</span>

In [68]:
df_customer = df_customer.rename(columns={
    'Customer_Name': 'customer_name',
    'Customer_Category': 'customer_category'
})


In [69]:
# Load existing IDs from DB
existing = pd.read_sql("SELECT customer_id FROM retail.dim_customer", engine)

existing_ids = set(existing['customer_id'])

df_new = df_customer[~df_customer['customer_id'].isin(existing_ids)]

print("Inserting new customers:", len(df_new))

df_new.to_sql(
    'dim_customer',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("dim_customer uploaded ✅")


Inserting new customers: 676645
dim_customer uploaded ✅


### <span style="color:#36A6A6">Converting dim_Product columns into lower case and uploading to PostgreSQL</span>

In [70]:
# Converting dim_Product columns into lower case and Uploading to PostgreSQL_DB
df_product.columns = df_product.columns.str.lower()

df_product.to_sql(
    'dim_product',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)
print("dim_product uploaded sucessfully✅")

dim_product uploaded sucessfully✅


In [ ]:
# Load existing IDs from DB
existing = pd.read_sql("SELECT product_id FROM retail.dim_product", engine)

existing_ids = set(existing['product_id'])

df_new = df_product[~df_product['product_id'].isin(existing_ids)]

print("Inserting new products:", len(df_new))

df_new.to_sql(
    'dim_product',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("dim_product uploaded successfully ✅")


### <span style="color:#36A6A6">Converting dim_Store columns into lower case and uploading to PostgreSQL</span>

In [71]:
# Converting dim-store columns into lower case and uploading to PostgreSQL_DB
df_store.columns=df_store.columns.str.lower()

df_store.to_sql(
    'dim_store',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("dim_store uploaded successfully✅")

dim_store uploaded successfully✅


In [ ]:
# Load existing IDs from DB
existing = pd.read_sql("SELECT store_id FROM retail.dim_store", engine)

existing_ids = set(existing['store_id'])

df_new = df_store[~df_store['store_id'].isin(existing_ids)]

print("Inserting new stores:", len(df_new))

df_new.to_sql(
    'dim_store',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("dim_store uploaded successfully ✅")


### <span style="color:#36A6A6">Converting dim_Date columns into lower case and uploading to PostgreSQL</span>

In [72]:
# Always rebuild df_date cleanly first

df_date = (
    df_exploded[['Date', 'Season']]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_date['Date'] = pd.to_datetime(df_date['Date'])

df_date['date_id'] = df_date['Date'].dt.strftime('%Y%m%d').astype(int)
df_date['year']     = df_date['Date'].dt.year
df_date['month']    = df_date['Date'].dt.month
df_date['month_name'] = df_date['Date'].dt.month_name()
df_date['day']      = df_date['Date'].dt.day
df_date['day_name'] = df_date['Date'].dt.day_name()

df_date.head()


,Date,Season,date_id,year,month,month_name,day,day_name
0,2022-01-21 06:27:29,Winter,20220121,2022,1,January,21,Friday
1,2023-03-01 13:01:21,Fall,20230301,2023,3,March,1,Wednesday
2,2024-03-21 15:37:04,Winter,20240321,2024,3,March,21,Thursday
3,2020-10-31 09:59:47,Spring,20201031,2020,10,October,31,Saturday
4,2020-12-10 00:59:59,Winter,20201210,2020,12,December,10,Thursday


In [73]:
existing_dates = pd.read_sql(
    "SELECT date_id FROM retail.dim_date",
    engine
)

existing_date_ids = set(existing_dates['date_id'])

missing_dates = (
    df_fact_final.loc[
        ~df_fact_final['date_id'].isin(existing_date_ids),
        'date_id'
    ]
    .drop_duplicates()
    .sort_values()
)

print("Missing date_ids:", missing_dates.head(20))
print("Total missing:", len(missing_dates))


Missing date_ids: 4137     20200101
784      20200102
67       20200103
2109     20200104
5032     20200105
1269     20200106
920      20200107
16453    20200108
2204     20200109
1585     20200110
2057     20200111
1473     20200112
3481     20200113
10556    20200114
295      20200115
32782    20200116
6639     20200117
1337     20200118
17143    20200119
1248     20200120
Name: date_id, dtype: int32
Total missing: 1600


In [74]:
df_missing_dates = pd.DataFrame({
    "date_id": missing_dates
})

df_missing_dates['date'] = pd.to_datetime(
    df_missing_dates['date_id'].astype(str),
    format="%Y%m%d"
)

df_missing_dates['year'] = df_missing_dates['date'].dt.year
df_missing_dates['month'] = df_missing_dates['date'].dt.month
df_missing_dates['day'] = df_missing_dates['date'].dt.day
df_missing_dates['month_name'] = df_missing_dates['date'].dt.month_name()
df_missing_dates['day_name'] = df_missing_dates['date'].dt.day_name()



In [75]:
# Add month_name and day_name columns to dim_date using Python + SQLAlchemy
with engine.begin() as conn:
    conn.exec_driver_sql("""
        ALTER TABLE retail.dim_date
        ADD COLUMN IF NOT EXISTS month_name TEXT;
    """)
    
    conn.exec_driver_sql("""
        ALTER TABLE retail.dim_date
        ADD COLUMN IF NOT EXISTS day_name TEXT;
    """)

print("Columns month_name and day_name added to retail.dim_date ✅")


Columns month_name and day_name added to retail.dim_date ✅


In [76]:
# STEP — Update month_name and day_name in the database
with engine.begin() as conn:
    conn.exec_driver_sql("""
        UPDATE retail.dim_date
        SET 
            month_name = TRIM(TO_CHAR(date, 'Month')),
            day_name   = TRIM(TO_CHAR(date, 'Day'));
    """)
    
print("dim_date month_name and day_name updated successfully ✅")


dim_date month_name and day_name updated successfully ✅


In [78]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ============================
# Database connection details
# ============================
DB_USER = "postgres"
DB_PASSWORD = "PAMPostG18"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Retail_Sales_DB"

# ============================
# Create SQLAlchemy engine
# ============================
url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(url)

# ============================
# Step 1: Add columns if missing
# ============================
add_columns_sql = """
ALTER TABLE retail.dim_date
ADD COLUMN IF NOT EXISTS day_abbr VARCHAR(3),
ADD COLUMN IF NOT EXISTS month_abbr VARCHAR(3);
"""

# ============================
# Step 2: Populate the columns
# ============================
update_columns_sql = """
UPDATE retail.dim_date
SET
    day_abbr   = LEFT(day_name, 3),
    month_abbr = LEFT(month_name, 3)
WHERE day_name IS NOT NULL
  AND month_name IS NOT NULL;
"""

# ============================
# Execute safely
# ============================
with engine.begin() as conn:
    conn.execute(text(add_columns_sql))
    conn.execute(text(update_columns_sql))

print("✅ dim_date updated with day_abbr and month_abbr columns successfully")


✅ dim_date updated with day_abbr and month_abbr columns successfully


In [79]:
df_date.head()

,Date,Season,date_id,year,month,month_name,day,day_name
0,2022-01-21 06:27:29,Winter,20220121,2022,1,January,21,Friday
1,2023-03-01 13:01:21,Fall,20230301,2023,3,March,1,Wednesday
2,2024-03-21 15:37:04,Winter,20240321,2024,3,March,21,Thursday
3,2020-10-31 09:59:47,Spring,20201031,2020,10,October,31,Saturday
4,2020-12-10 00:59:59,Winter,20201210,2020,12,December,10,Thursday


In [80]:
df_missing_dates.to_sql(
    'dim_date',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("✅ Missing dates inserted into dim_date")


✅ Missing dates inserted into dim_date


In [81]:
# Load existing IDs from PostgreSQL
existing = pd.read_sql("SELECT date_id FROM retail.dim_date", engine)

existing_ids = set(existing['date_id'])

# Filter out dates already uploaded
df_new = df_date[~df_date['date_id'].isin(existing_ids)]

print("New rows to upload:", len(df_new))

# Upload only missing rows
df_new_upload = df_new.rename(columns={
    'Date': 'date',
    'Season': 'season'
})

df_new_upload['date'] = pd.to_datetime(df_new_upload['date']).dt.date

df_new_upload.to_sql(
    'dim_date',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("dim_date updated successfully!")


New rows to upload: 0
dim_date updated successfully!


In [97]:
df_date.head()

,Date,Season,date_id,year,month,month_name,day,day_name
0,2022-01-21 06:27:29,Winter,20220121,2022,1,January,21,Friday
1,2023-03-01 13:01:21,Fall,20230301,2023,3,March,1,Wednesday
2,2024-03-21 15:37:04,Winter,20240321,2024,3,March,21,Thursday
3,2020-10-31 09:59:47,Spring,20201031,2020,10,October,31,Saturday
4,2020-12-10 00:59:59,Winter,20201210,2020,12,December,10,Thursday


### <span style="color:#36A6A6">Converting dim_Promotion columns into lower case and uploading to PostgreSQL</span>

In [82]:
# Ensure columns are lowercase (optional but consistent)
df_promotion.columns = df_promotion.columns.str.lower()

# 1 — Load existing promotion_ids from PostgreSQL
existing = pd.read_sql("SELECT promotion_id FROM retail.dim_promotion", engine)

existing_ids = set(existing['promotion_id'])

# 2 — Filter only new rows (NOT already in DB)
df_new_promos = df_promotion[~df_promotion['promotion_id'].isin(existing_ids)]

print("New promotions to upload:", len(df_new_promos))

# 3 — Upload only the new rows
if len(df_new_promos) > 0:
    df_new_promos.to_sql(
        'dim_promotion',
        engine,
        schema='retail',
        if_exists='append',
        index=False
    )
    print("dim_promotion updated successfully ✔")
else:
    print("No new promotion rows to insert — table already up to date.")


New promotions to upload: 3
dim_promotion updated successfully ✔


### <span style="color:#36A6A6">Converting dim_Payment_Method columns into lower case and uploading to PostgreSQL</span>

In [89]:
# Load existing IDs from DB
existing = pd.read_sql("SELECT payment_method_id FROM retail.dim_payment_method", engine)

existing_ids = set(existing['payment_method_id'])

df_new = df_payment_method[~df_payment_method['payment_method_id'].isin(existing_ids)]

print("Inserting new payment_methods:", len(df_new))

df_new.to_sql(
    'dim_payment_method',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("dim_payment_method uploaded successfully ✅")


Inserting new payment_methods: 0
dim_payment_method uploaded successfully ✅


In [84]:
def safe_dim_upload(df, table_name, key_column, engine, schema='retail'):
    existing = pd.read_sql(f"SELECT {key_column} FROM {schema}.{table_name}", engine)
    existing_ids = set(existing[key_column])
    
    df_new = df[~df[key_column].isin(existing_ids)]
    
    print(f"{table_name}: new rows to upload = {len(df_new)}")
    
    if len(df_new) > 0:
        df_new.to_sql(
            table_name,
            engine,
            schema=schema,
            if_exists='append',
            index=False
        )
        print(f"{table_name} updated ✔")
    else:
        print(f"{table_name} already up-to-date ✔")


### <span style="color:#36A6A6">Converting Fact_Sales columns into lower case and uploading to PostgreSQL</span>

In [85]:
df_fact_final.columns = df_fact_final.columns.str.lower()


In [87]:
df_payment_method = df_payment_method.rename(columns={
    'Payment_Method': 'payment_method'
})


In [88]:
existing = pd.read_sql(
    "SELECT payment_method_id FROM retail.dim_payment_method",
    engine
)

existing_ids = set(existing['payment_method_id'])

df_new = df_payment_method[
    ~df_payment_method['payment_method_id'].isin(existing_ids)
]

df_new.to_sql(
    'dim_payment_method',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)

print("✅ dim_payment_method uploaded successfully")


✅ dim_payment_method uploaded successfully


In [90]:
df_fact_final.to_sql(
    'fact_sales',
    engine,
    schema='retail',
    if_exists='append',
    index=False
)
print("✅ fact_sales uploaded successfully!")


✅ fact_sales uploaded successfully!


### <span style="color:#36A6A6">Running a quick DataBase checks from Python</span>

In [91]:
# STEP 11 — Quick DB checks from Python
with engine.connect() as conn:
    print(conn.exec_driver_sql("SELECT count(*) FROM retail.dim_product").scalar(), 
          "products in DB")
    print(conn.exec_driver_sql("SELECT count(*) FROM retail.dim_customer").scalar(), 
          "customers in DB")
    print(conn.exec_driver_sql("SELECT count(*) FROM retail.fact_sales").scalar(), 
          "fact rows in DB")
    # check referential integrity approximations
    print("Missing product FK count:",
          conn.exec_driver_sql("""
            SELECT COUNT(*) FROM retail.fact_sales f
            LEFT JOIN retail.dim_product p ON f.product_id = p.product_id
            WHERE p.product_id IS NULL
          """).scalar())


81 products in DB
676645 customers in DB
3016917 fact rows in DB
Missing product FK count: 0


### <span style="color:#36A6A6">Add month_name and day_name coloumns to dim_date using Python + SQLAlchemy</span>

In [93]:
# Add month_name and day_name columns to dim_date using Python + SQLAlchemy
with engine.begin() as conn:
    conn.exec_driver_sql("""
        ALTER TABLE retail.dim_date
        ADD COLUMN IF NOT EXISTS month_name TEXT;
    """)
    
    conn.exec_driver_sql("""
        ALTER TABLE retail.dim_date
        ADD COLUMN IF NOT EXISTS day_name TEXT;
    """)

print("Columns month_name and day_name added to retail.dim_date ✅")


Columns month_name and day_name added to retail.dim_date ✅


### <span style="color:#36A6A6">Update month_name and day_name in the database</span>

In [94]:
# STEP — Update month_name and day_name in the database
with engine.begin() as conn:
    conn.exec_driver_sql("""
        UPDATE retail.dim_date
        SET 
            month_name = TRIM(TO_CHAR(date, 'Month')),
            day_name   = TRIM(TO_CHAR(date, 'Day'));
    """)
    
print("dim_date month_name and day_name updated successfully ✅")


dim_date month_name and day_name updated successfully ✅


## <span style = "color:#36A6A6;">Some Real-World Business Analysis Questions Using Pandas Only (Python)</span>

### <span style=color:#36A6A6>Question 1: Which store has the highest total sales revenue?</span>

In [116]:
# Calculate revenue
df_fact['revenue'] = df_fact['line_quantity'] * df_fact['Total_Cost']

# Merge with store table to get city and store_type
df_merged = df_fact.merge(df_store, on='store_id', how='left')

# Aggregate revenue by store
store_revenue = df_merged.groupby(['store_id', 'city', 'store_type'])['revenue'].sum().sort_values(ascending=False)
print(store_revenue.head(10))


store_id  city           store_type       
29        Chicago        Convenience Store    2699391.99
53        Boston         Pharmacy             2688342.27
13        Seattle        Warehouse Club       2682615.84
10        Boston         Department Store     2679835.80
56        New York       Supermarket          2679268.33
43        Dallas         Department Store     2677745.99
17        San Francisco  Department Store     2676010.38
30        Dallas         Supermarket          2672301.05
57        Chicago        Warehouse Club       2671923.98
47        Atlanta        Warehouse Club       2670922.31
Name: revenue, dtype: float64


### <span style="color:#36A6A6">Top 5 cities by total sales revenue</span>

In [117]:
city_revenue = df_merged.groupby('city')['revenue'].sum().sort_values(ascending=False).head(5)
print(city_revenue)


city
Chicago          15913200.25
Boston           15904253.14
Dallas           15878465.59
New York         15854814.52
San Francisco    15846133.99
Name: revenue, dtype: float64


### <span style="color:#36A6A6">Which store type performs best (highest average revenue per transaction)?</span>

In [118]:
avg_revenue_by_type = df_merged.groupby('store_type')['revenue'].mean().sort_values(ascending=False)
print(avg_revenue_by_type)


store_type
Warehouse Club       52.610219
Pharmacy             52.541703
Supermarket          52.475582
Convenience Store    52.439062
Specialty Store      52.413825
Department Store     52.369478
Name: revenue, dtype: float64


### <span style="color:#36A6A6">Monthly sales trend per store</span>

### <span style="color:#36A6A6">Product category performance per store type</span>

In [120]:
df_merged = df_merged.merge(df_product, on='product_id', how='left')

category_store = df_merged.groupby(['store_type'])['revenue'].sum().sort_values(ascending=False)
print(category_store)


store_type
Warehouse Club       26493611.93
Pharmacy             26446708.74
Supermarket          26424813.80
Convenience Store    26396513.05
Department Store     26355620.74
Specialty Store      26195643.46
Name: revenue, dtype: float64


### <span style="color:#36A6A6">Best customers per store</span>

In [121]:
top_customers_store = df_merged.groupby(['store_id', 'customer_id'])['revenue'].sum().sort_values(ascending=False)
print(top_customers_store.head(10))


store_id  customer_id
23        606            1344.76
57        6333           1312.90
55        60626          1295.54
23        14859          1176.78
13        9551           1175.28
1         4327           1165.29
36        88868          1147.20
47        65349          1132.70
30        120524         1112.95
11        11313          1067.98
Name: revenue, dtype: float64


In [122]:
print(df_fact.columns)


Index(['Transaction_ID', 'Date', 'Customer_Name', 'Product', 'Total_Items',
       'Total_Cost', 'Payment_Method', 'City', 'Store_Type',
       'Discount_Applied', 'Customer_Category', 'Season', 'Promotion',
       'product_count', 'line_revenue', 'line_quantity', 'customer_id',
       'product_id', 'store_id', 'date_id', 'promotion_id',
       'payment_method_id', 'revenue'],
      dtype='object')


In [95]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ============================
# Database connection details
# ============================
DB_USER = "postgres"
DB_PASSWORD = "PAMPostG18"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Retail_Sales_DB"

# ============================
# Create SQLAlchemy engine
# ============================
url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(url)

# ============================
# Step 1: Add columns if missing
# ============================
add_columns_sql = """
ALTER TABLE retail.dim_date
ADD COLUMN IF NOT EXISTS day_abbr VARCHAR(3),
ADD COLUMN IF NOT EXISTS month_abbr VARCHAR(3);
"""

# ============================
# Step 2: Populate the columns
# ============================
update_columns_sql = """
UPDATE retail.dim_date
SET
    day_abbr   = LEFT(day_name, 3),
    month_abbr = LEFT(month_name, 3)
WHERE day_name IS NOT NULL
  AND month_name IS NOT NULL;
"""

# ============================
# Execute safely
# ============================
with engine.begin() as conn:
    conn.execute(text(add_columns_sql))
    conn.execute(text(update_columns_sql))

print("✅ dim_date updated with day_abbr and month_abbr columns successfully")


✅ dim_date updated with day_abbr and month_abbr columns successfully


In [100]:
df_date.isnull().sum()


date_id       0
Date          0
year          0
month         0
month_name    0
day           0
day_name      0
season        0
dtype: int64

In [109]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'


In [107]:
df_date = (
    df_exploded[['Date']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Ensure datetime
df_date['Date'] = pd.to_datetime(df_date['Date'])

# Surrogate key (YYYYMMDD)
df_date['date_id'] = df_date['Date'].dt.strftime('%Y%m%d').astype(int)

# Date parts
df_date['year'] = df_date['Date'].dt.year
df_date['month'] = df_date['Date'].dt.month
df_date['day'] = df_date['Date'].dt.day

# Names
df_date['month_name'] = df_date['Date'].dt.month_name()
df_date['day_name'] = df_date['Date'].dt.day_name()

# Season (derived, NEVER NULL)
df_date['season'] = df_date['month'].apply(get_season)

# Final order
df_date = df_date[
    ['date_id', 'Date', 'year', 'month', 'month_name',
     'day', 'day_name', 'season']
]

df_date.head()


,date_id,Date,year,month,month_name,day,day_name,season
0,20220121,2022-01-21 06:27:29,2022,1,January,21,Friday,Winter
1,20230301,2023-03-01 13:01:21,2023,3,March,1,Wednesday,Spring
2,20240321,2024-03-21 15:37:04,2024,3,March,21,Thursday,Spring
3,20201031,2020-10-31 09:59:47,2020,10,October,31,Saturday,Autumn
4,20201210,2020-12-10 00:59:59,2020,12,December,10,Thursday,Winter


In [114]:
import pandas as pd

for table in ["retail.fact_sales","retail.dim_customer","retail.dim_date",
              "retail.dim_payment_method","retail.dim_product","retail.dim_promotion",
              "retail.dim_store"]:
    print(f"\n--- {table.upper()} ---")
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine))



--- RETAIL.FACT_SALES ---


,fact_id,transaction_id,date_id,customer_id,product_id,store_id,promotion_id,payment_method_id,discount_applied,quantity,revenue
0,1001,1000000000,20220121,1,1,1,1,1,True,1,23.88
1,1002,1000000000,20220121,1,2,1,1,1,True,1,23.88
2,1003,1000000000,20220121,1,3,1,1,1,True,1,23.88
3,1004,1000000001,20230301,2,4,2,2,2,True,1,5.19
4,1005,1000000001,20230301,2,5,2,2,2,True,1,5.19



--- RETAIL.DIM_CUSTOMER ---


,customer_id,customer_name,customer_category
0,1,Stacey Price,Homemaker
1,2,Michelle Carlson,Professional
2,3,Lisa Graves,Professional
3,4,Mrs. Patricia May,Homemaker
4,5,Susan Mitchell,Young Adult



--- RETAIL.DIM_DATE ---


...

,product_id,product
0,1,Ketchup
1,2,Shaving Cream
2,3,Light Bulbs
3,4,Ice Cream
4,5,Milk



--- RETAIL.DIM_PROMOTION ---


,promotion_id,promotion
0,1,No Promotion
1,2,BOGO (Buy One Get One)
2,3,Discount on Selected Items



--- RETAIL.DIM_STORE ---


,store_id,city,store_type
0,1,Los Angeles,Warehouse Club
1,2,San Francisco,Specialty Store
2,3,Houston,Department Store
3,4,Chicago,Pharmacy
4,5,Houston,Specialty Store


In [112]:
# 1. Check duplicates on natural key
print("Duplicate dates:", df_date['Date'].duplicated().sum())

# 2. Check nulls
print(df_date.isnull().sum())

# 3. Check date range
print("Min date:", df_date['Date'].min())
print("Max date:", df_date['Date'].max())

# 4. Sample rows
df_date.head()


Duplicate dates: 0
date_id       0
Date          0
year          0
month         0
month_name    0
day           0
day_name      0
season        0
dtype: int64
Min date: 2020-01-01 00:03:54
Max date: 2024-05-18 19:31:03


,date_id,Date,year,month,month_name,day,day_name,season
0,20220121,2022-01-21 06:27:29,2022,1,January,21,Friday,Winter
1,20230301,2023-03-01 13:01:21,2023,3,March,1,Wednesday,Spring
2,20240321,2024-03-21 15:37:04,2024,3,March,21,Thursday,Spring
3,20201031,2020-10-31 09:59:47,2020,10,October,31,Saturday,Autumn
4,20201210,2020-12-10 00:59:59,2020,12,December,10,Thursday,Winter
